<a href="https://colab.research.google.com/github/herinishere/Text-Summarizer/blob/main/text_summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pytextrank spacy
!python -m spacy download en_core_web_sm

In [ ]:
import torch
import nltk
import spacy
import pytextrank

In [ ]:
nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("textrank")

In [ ]:
from datasets import load_dataset
train = load_dataset("NortheasternUniversity/big_patent","all",split="train[:2000]")
val = load_dataset("NortheasternUniversity/big_patent","all",split="validation[:100]")

In [ ]:
from transformers import T5Tokenizer,T5ForConditionalGeneration
tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
embedder = SentenceTransformer("all-MiniLM-L6-v2")
def summary(text,maxi=8,lambda_param=0.7):
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents]
    if len(sentences) <= maxi:
        return " ".join(sentences)
    embeddings = embedder.encode(sentences)
    embeddings1 = embedder.encode([text])
    doc_sim = cosine_similarity(
        embeddings,
        embeddings1
    ).flatten()
    sentence = []
    idx = []
    idx1 = np.argmax(doc_sim)
    sentence.append(sentences[idx1])
    idx.append(idx1)
    while len(sentence) < maxi:
        scores = []
        for i in range(len(sentences)):
            if i in idx:
                scores.append(-1)
                continue
            new_sim = cosine_similarity(
                embeddings[i].reshape(1, -1),
                embeddings[idx]
            ).max()
            mmr = (
                lambda_param*doc_sim[i]-(1-lambda_param)*new_sim
            )
            scores.append(mmr)
        next_idx = np.argmax(scores)
        sentence.append(sentences[next_idx])
        idx.append(next_idx)
    return " ".join(sentence)

In [ ]:
def preprocess(examples):
    txt = [
        summary(text, maxi=8)
        for text in examples["description"]
    ]
    inputs = tokenizer(
        ["summarize: " + t for t in txt],
        max_length=512,
        truncation=True
    )
    labels = tokenizer(
        examples["abstract"],
        max_length=160,
        truncation=True
    )
    inputs["labels"] = labels["input_ids"]
    return inputs

In [ ]:
train = train.map(
    preprocess,
    batched=True,
)

In [ ]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [ ]:
from transformers import TrainingArguments,Trainer
training_args = TrainingArguments(
    output_dir="./txt_summarizer",
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    num_train_epochs=3,
    save_strategy="epoch",
    report_to="none",
    fp16=False
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train,
    tokenizer=tokenizer,
    data_collator=data_collator
)
trainer.train()

In [ ]:
saving= "./my_trained_t5_summarizer"
model.save_pretrained(saving)
tokenizer.save_pretrained(saving)
print("Model and tokenizer saved!")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

In [ ]:
def summarize(text):
    extracted = summary(text, maxi=8)
    inp = tokenizer(
        "summarize: " + extracted,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    inp = {k:v.to(device) for k,v in inp.items()}
    out = model.generate(
        inp["input_ids"],
        max_length=160,
        num_beams=4,
        no_repeat_ngram_size=3
    )
    return tokenizer.decode(
        out[0],
        skip_special_tokens=True
    )

In [ ]:
!pip install -q bert-score
from bert_score import score as bertscore

In [ ]:
preds = []
ref = []
for sample in val:
    pred = summarize(sample["description"])
    preds.append(pred)
    ref.append(sample["abstract"])
P, R, F1 = bertscore(preds,ref,lang="en")
print(f"BERTScore Precision:{P.mean().item():.4f}")
print(f"BERTScore Recall:{R.mean().item():.4f}")
print(f"BERTScore F1:{F1.mean().item():.4f}")